### This notebook compute "VSS6. Volume of annual water demand for population use" indicator for the 27 basins of IKI Project

Spanish: Volumen de la demanda anual del agua para uso poblacional

**Created:** 12/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 12/22/2025

**Status:** Complete (for baseline scenario)

**QA Status:** reviewed by Scott Sheeder  

**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Vulnerabilidad

**Packages:** pandas, sqlite3, scipy, tqdm  
 
**Inputs:**  waterALLOC output database

**Outputs:** 
 
**Assumptions:** 

**N/A Handling:** 
 
**Future work:** 
 
**Notes:** Here we directly query the waterALLOC output SQL database and import the results to the indicators database. We have included some code to check the min and max values of the results and the min and max values listed in the indicators database to ensure they align. Code is provided to update the min and max values in the indicators DB as needed. 

General Methodology:  
1. Queary SQL for scenario average annual water supply and demand for population use.  
2. Check that min and max values match the expected range based on the Indicators Table.  
3. Insert demand into the SQL database.  

In [1]:
#import numpy as np
import pandas as pd
import sqlite3
#import geopandas as gpd
from scipy.spatial import cKDTree
#import os
from tqdm import tqdm

In [2]:
# set up user and database path
#user = 'jmayo'
#user= 'cpickering'
#user = 'sgilson'
#user = 'nreynolds'
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [6]:
# set up indicator ID and get scenarios from database
IndID = 406 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
conn = sqlite3.connect(db_path)

scenarios_df = pd.read_sql_query(
    """
    SELECT ScnID, WaScnName
    FROM ScnMod
    """,
    conn
)

conn.close()

# For now: only baseline and first future
#scenario_ids = scenarios_df.loc[
#    scenarios_df['ScnID'].isin([1, 2]), 'ScnID'
#].tolist()

# for all scenarios:
scenario_ids = scenarios_df['ScnID'].tolist()

In [15]:
## check what scenarios are available in the WaterALLOC database
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)

# Query available scenarios
scenarios_query = """
SELECT DISTINCT Scenario
FROM Scenarios
ORDER BY Scenario
"""

wa_scenarios_df = pd.read_sql_query(scenarios_query, conn_wa)

print("Available scenarios in WaterALLOC DB:")
for s in wa_scenarios_df["Scenario"]:
    print(f" - {s}")

Available scenarios in WaterALLOC DB:
 - CC_CMIP6_85_2050
 - Linea_Base_2020
 - Linea_Base_2020_Embalses


In [13]:
# for now only do the baseline and first future scenario (update in future based on structure of new scenarios table)
target_scenarios = ["Linea_Base_2020","CC_CMIP6_85_2050"]
scenarios_subset = scenarios_df[scenarios_df["WaScnName"].isin(target_scenarios)][["ScnID", "WaScnName"]]

In [ ]:
dfs = []

query_demanda = """
SELECT 
    a.comid AS COMID,
    a.[Demanda] AS Demanda,
    a.[Suministro] AS Suministro
FROM "WAMSS_Demanda anual promedio por tipo de demanda por COMID" AS a
JOIN WAMMS_RunsInfo AS b 
    ON a.RunID = b.RunID
JOIN Scenarios AS c 
    ON c.ScnID = b.ScnID
WHERE c.Scenario = ?
  AND a.Sector = ?
"""

sector_name = "Poblacional"

for _, row in scenarios_subset.iterrows():
    scn_id = row["ScnID"]
    scenario_name = row["WaScnName"]

    df = pd.read_sql_query(
        query_demanda,
        conn_wa,
        params=(scenario_name, sector_name)
    )

    df["ScnID"] = scn_id
    df["Scenario"] = scenario_name

    dfs.append(df)

# Combine all scenarios
demanda_df = pd.concat(dfs, ignore_index=True)

# Close connection
conn_wa.close()

In [17]:
demanda_df

,COMID,Demanda,Suministro,ScnID,Scenario
0,304421300,6.46,6.460000,1,Linea_Base_2020
1,304598600,1.79,1.790000,1,Linea_Base_2020
2,304640700,44.27,44.270000,1,Linea_Base_2020
3,304677500,1.62,1.620000,1,Linea_Base_2020
4,304698200,25.67,25.670000,1,Linea_Base_2020
...,...,...,...,...,...
113,310712000,113.86,113.129737,2,CC_CMIP6_85_2050
114,310723800,20.47,18.582105,2,CC_CMIP6_85_2050
115,310726200,7.90,7.853947,2,CC_CMIP6_85_2050
116,310726300,8.85,7.992105,2,CC_CMIP6_85_2050


In [19]:
# insert data into the sqlite database
# Connect to SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

rows_to_insert = []
for _, row in demanda_df.iterrows():
    rows_to_insert.append((row['ScnID'], IndID, row['COMID'], row['Demanda']))

# Insert data into IndValues_Dyn
insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""

In [20]:
# Check that min and max values match the expected range based on the Indicators Table

indicator_limits = pd.read_sql_query(
    """
    SELECT IndID, Min, Max
    FROM Indicators
    WHERE IndID = ?
    """,
    conn,
    params=(IndID,)
)

if indicator_limits.empty:
    raise ValueError(f"No entry found in Indicators table for IndID = {IndID}")

ind_min = indicator_limits.loc[0, 'Min']
ind_max = indicator_limits.loc[0, 'Max']

print(f"\nIndicator {IndID} limits from Indicators table -> Min: {ind_min}, Max: {ind_max}")

# Compute value stats by scenario
value_stats = demanda_df.groupby('ScnID')['Demanda'].agg(['min', 'max', 'count']).reset_index()
print("\n=== Values to be inserted (by scenario) ===")
print(value_stats)

# Check for duplicates in the rows to insert
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("\nDuplicates in rows_to_insert:")
print(df_check[duplicates])


Indicator 406 limits from Indicators table -> Min: 0, Max: 50000

=== Values to be inserted (by scenario) ===
   ScnID   min      max  count
0      1  0.95  43523.3     59
1      2  0.95  43523.3     59

Duplicates in rows_to_insert:
Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [21]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()